In [ ]:
# =========================
# Program: CNN Klasik (4-layer) + eval + debugging
# Run in Google Colab (recommended) or local (adjust paths)
# =========================

# 0) Install deps (Colab usually has these; uncomment if missing)
# !pip install -q scikit-learn seaborn tqdm

# 1) Imports
import os, shutil, time, zipfile, math
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, utils

from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import label_binarize

# 2) Paths & config
# If using Colab, mount Drive first (uncomment)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_ZIP = "/content/drive/MyDrive/hama/hama1_split.zip"   # your path (you said this)
LOCAL_ZIP = "/content/hama1_split.zip"
EXTRACT_TO = "/content/dataset/hama/hama_split"             # final extracted folder containing train/valid/test
MODEL_SAVE_DIR = "/content/drive/MyDrive/hama/models_cnn_classic"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# Training config (tweak as needed)
BATCH_SIZE = 64       # for CPU lower if memory limited (32)
CROP_SIZE = 64
NUM_WORKERS = 2       # on Colab 2 is safe; on local increase to CPU cores-1
MAX_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 8
LR = 1e-3
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 3) Copy ZIP from Drive -> local and extract (fast)
if not os.path.exists(LOCAL_ZIP):
    print("Copying ZIP from Drive to local...")
    shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)
else:
    print("ZIP already exists locally:", LOCAL_ZIP)

# Extract
if os.path.exists(EXTRACT_TO):
    print("Extraction folder already exists:", EXTRACT_TO)
else:
    print("Extracting ZIP to:", EXTRACT_TO)
    with zipfile.ZipFile(LOCAL_ZIP, 'r') as z:
        z.extractall(EXTRACT_TO)
    print("Extract done.")

# Verify structure: expect EXTRACT_TO/train, /valid, /test
for sub in ["train", "valid", "test"]:
    p = os.path.join(EXTRACT_TO, sub)
    if not os.path.isdir(p):
        raise FileNotFoundError(f"Expected folder not found: {p}. Check ZIP structure.")
print("Dataset structure OK.")

# 4) Transforms & Datasets
train_transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(CROP_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])
eval_transform = transforms.Compose([
    transforms.Resize(int(CROP_SIZE*1.1)),
    transforms.CenterCrop(CROP_SIZE),
    transforms.ToTensor()
])

train_dir = os.path.join(EXTRACT_TO, "train")
valid_dir = os.path.join(EXTRACT_TO, "valid")
test_dir  = os.path.join(EXTRACT_TO, "test")

train_set = datasets.ImageFolder(train_dir, transform=train_transform)
valid_set = datasets.ImageFolder(valid_dir, transform=eval_transform)
test_set  = datasets.ImageFolder(test_dir, transform=eval_transform)

classes = train_set.classes
num_classes = len(classes)
print("Classes:", classes)
print("Num classes:", num_classes)

# 5) Print per-class counts (debug)
def class_counts_from_samples(samples):
    labels = [s[1] for s in samples]
    c = Counter(labels)
    return [c[i] for i in range(len(classes))]

print("Train counts:", class_counts_from_samples(train_set.samples))
print("Valid counts:", class_counts_from_samples(valid_set.samples))
print("Test counts:",  class_counts_from_samples(test_set.samples))

# 6) Create WeightedRandomSampler & class weights for loss (to mitigate imbalance)
train_counts = class_counts_from_samples(train_set.samples)
class_weights = torch.tensor([1.0/(c if c>0 else 1.0) for c in train_counts], dtype=torch.float)
class_weights = class_weights / class_weights.sum() * num_classes   # normalized (optional)
class_weights = class_weights.to(device)
print("Class weights (loss):", class_weights.cpu().numpy())

sample_weights = [1.0 / train_counts[label] for _, label in train_set.samples]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# 7) Define Classic CNN (4-layer conv) — similar to your earlier model but explicit
class ClassicCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # conv blocks: Conv->ReLU->MaxPool
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )  # 64 -> 32

        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )  # 32 -> 16

        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )  # 16 -> 8

        self.conv4 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )  # 8 -> 4

        self.flatten_dim = 128 * 4 * 4  # if input 64x64
        self.classifier = nn.Sequential(
            nn.Linear(self.flatten_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = ClassicCNN(num_classes).to(device)
print(model)
print("Total params:", sum(p.numel() for p in model.parameters()))

# 8) Loss, optimizer, scheduler
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)

# 9) Training utilities & diagnostics hooks
def compute_metrics(y_true, y_pred, labels, target_names):
    report = classification_report(y_true, y_pred, labels=labels, target_names=target_names, zero_division=0, output_dict=True)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    return report, cm

def evaluate(model, loader, device):
    model.eval()
    ys = []
    preds = []
    losses = []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device); y = y.to(device)
            out = model(X)
            loss = criterion(out, y)
            losses.append(loss.item() * X.size(0))
            p = out.argmax(1).cpu().numpy()
            preds.extend(p.tolist())
            ys.extend(y.cpu().numpy().tolist())
    avg_loss = sum(losses)/len(loader.dataset)
    acc = (np.array(ys) == np.array(preds)).mean()
    f1_macro = f1_score(ys, preds, average='macro', zero_division=0)
    return avg_loss, acc, f1_macro, ys, preds

# hooks: capture first conv activations for debugging visualizations
activation = {}
def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach().cpu()
    return hook
model.conv1.register_forward_hook(get_activation('conv1'))

# 10) Training loop with early stopping on val F1
best_val_f1 = 0.0
patience = 0
train_losses, val_losses = [], []
train_accs, val_accs = [], []
start_time = time.time()

for epoch in range(1, MAX_EPOCHS+1):
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0
    # track grad norms and weight hist occasionally
    batch_times = []
    for X, y in tqdm(train_loader, desc=f"Epoch {epoch} training"):
        t0 = time.time()
        X, y = X.to(device), y.to(device)
        out = model(X)
        loss = criterion(out, y)
        optimizer.zero_grad()
        loss.backward()
        # debugging: gradient norm (first layer)
        grad_norm = 0.0
        for p in model.parameters():
            if p.grad is not None:
                grad_norm += p.grad.data.norm(2).item() ** 2
        grad_norm = math.sqrt(grad_norm)
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        running_correct += (out.argmax(1) == y).sum().item()
        total += X.size(0)
        batch_times.append(time.time() - t0)

    train_loss = running_loss / total
    train_acc = running_correct / total
    val_loss, val_acc, val_f1, _, _ = evaluate(model, valid_loader, device)

    train_losses.append(train_loss); val_losses.append(val_loss)
    train_accs.append(train_acc); val_accs.append(val_acc)

    print(f"Epoch {epoch:02d} | train_loss {train_loss:.4f} acc {train_acc:.4f} | val_loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f} | avg_batch_s {np.mean(batch_times):.3f}")

    # scheduler update monitors validation F1 via step with metric
    scheduler.step(val_f1)

    if val_f1 > best_val_f1 + 1e-4:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(MODEL_SAVE_DIR, "cnn_classic_best.pth"))
        print("  -> New best model saved.")
        patience = 0
    else:
        patience += 1

    if patience >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

end_time = time.time()
print("Training done in", round(end_time - start_time,1), "s")

# 11) Load best model and evaluate on test
best_path = os.path.join(MODEL_SAVE_DIR, "cnn_classic_best.pth")
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=device))
    print("Loaded best model.")
else:
    print("Best model not found - using last state.")

test_loss, test_acc, test_f1, ys_true, ys_pred = evaluate(model, test_loader, device)
print(f"\n=== TEST RESULTS ===\nLoss {test_loss:.4f} Acc {test_acc:.4f} MacroF1 {test_f1:.4f}")

# 12) Full classification report & confusion matrix
report_dict, cm = compute_metrics(ys_true, ys_pred, labels=list(range(num_classes)), target_names=classes)
report = pd.DataFrame(report_dict).transpose()
print("\nClassification report (dict -> table):")
display(report)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=classes, yticklabels=classes, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix (Test)")
plt.show()

# 13) Additional diagnostics & debugging visuals

# 13.1 Learning curves
plt.figure(figsize=(12,5))
plt.subplot(1,2,1); plt.plot(train_losses, label="train_loss"); plt.plot(val_losses, label="val_loss"); plt.legend(); plt.title("Loss")
plt.subplot(1,2,2); plt.plot(train_accs, label="train_acc"); plt.plot(val_accs, label="val_acc"); plt.legend(); plt.title("Accuracy")
plt.show()

# 13.2 Show some misclassified samples (up to 24)
def show_misclassified(n=24):
    bad_idx = [i for i,(t,p) in enumerate(zip(ys_true, ys_pred)) if t!=p]
    n = min(n, len(bad_idx))
    if n==0:
        print("No misclassified samples.")
        return
    picks = bad_idx[:n]
    imgs = []
    labels_true = []
    labels_pred = []
    # we need to iterate test_set to get images; use dataset.samples which aligns with loader ordering
    for idx in picks:
        path, true_label = test_set.samples[idx]
        img = test_set.loader(path)
        img = eval_transform(img) if isinstance(eval_transform, transforms.Compose) else eval_transform(img)
        imgs.append(img)
        labels_true.append(true_label)
        labels_pred.append(ys_pred[idx])
    grid = utils.make_grid(imgs, nrow=6, normalize=True, scale_each=True)
    plt.figure(figsize=(12,8))
    plt.imshow(np.transpose(grid.numpy(), (1,2,0)))
    plt.axis('off')
    titles = [f"T:{classes[t]} P:{classes[p]}" for t,p in zip(labels_true, labels_pred)]
    print("\n".join(titles[:n]))
show_misclassified(18)

# 13.3 Visualize first-conv feature maps for a few test images (debugging)
def visualize_activations(sample_idx=0, n_maps=8):
    model.eval()
    path, label = test_set.samples[sample_idx]
    img = test_set.loader(path)
    inp = eval_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        _ = model(inp)
    if 'conv1' in activation:
        acts = activation['conv1'].squeeze(0)  # shape [C,H,W] on cpu
        n_maps = min(n_maps, acts.shape[0])
        fig, axes = plt.subplots(1, n_maps, figsize=(n_maps*2,2))
        for i in range(n_maps):
            axes[i].imshow(acts[i].numpy(), cmap='viridis')
            axes[i].axis('off')
        plt.suptitle(f"First-conv feature maps for sample {sample_idx} (label={classes[label]})")
        plt.show()
    else:
        print("No activation captured.")
visualize_activations(0, n_maps=8)

# 13.4 Weight histograms of conv layers
def plot_weight_hists():
    names = ['conv1.0.weight','conv2.0.weight','conv3.0.weight','conv4.0.weight']
    plt.figure(figsize=(12,8))
    for i, name in enumerate(names):
        # locate param
        try:
            w = dict(model.named_parameters())[name].cpu().detach().numpy().ravel()
            plt.subplot(2,2,i+1)
            plt.hist(w, bins=50); plt.title(name)
        except KeyError:
            pass
    plt.tight_layout()
plot_weight_hists()

# 14) Save final artifacts (report CSV + confusion matrix image)
report.to_csv(os.path.join(MODEL_SAVE_DIR, "classification_report.csv"))
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=classes, yticklabels=classes, cmap="Blues")
plt.savefig(os.path.join(MODEL_SAVE_DIR, "confusion_matrix_test.png"), bbox_inches='tight')
print("Saved report & confusion matrix to", MODEL_SAVE_DIR)

# Optional: compute multiclass ROC-AUC (one-vs-rest) if feasible
try:
    y_true_bin = label_binarize(ys_true, classes=list(range(num_classes)))
    # Need model outputs (probabilities) for test set to compute ROC-AUC properly.
    # We'll get softmax probabilities:
    model.eval()
    probs = []
    with torch.no_grad():
        for X,y in test_loader:
            X = X.to(device)
            out = model(X)
            p = nn.functional.softmax(out, dim=1).cpu().numpy()
            probs.append(p)
    probs = np.vstack(probs)
    if probs.shape[0] != len(ys_true):
        print("Warning: probs rows vs true labels mismatch; skipping ROC-AUC")
    else:
        aucs = []
        for i in range(num_classes):
            try:
                auc = roc_auc_score(y_true_bin[:,i], probs[:,i])
            except Exception as e:
                auc = np.nan
            aucs.append(auc)
        auc_df = pd.DataFrame({'class':classes, 'roc_auc':aucs})
        print("\nPer-class ROC-AUC:")
        display(auc_df)
except Exception as e:
    print("ROC-AUC computation skipped due to:", e)

print("\nALL DONE. Model + artifacts saved at:", MODEL_SAVE_DIR)
